In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 04 · Variables — cada idea es una hipótesis que se mide

**Proyecto RiskVision · Grupo 6.** Dataset: *Default of Credit Card Clients* (UCI, CC BY 4.0).

## Para qué sirve este notebook
Inventar variables es fácil; saber si de verdad ayudan es lo difícil. Aquí cada idea se prueba igual: mismo modelo, mismos pliegues y misma métrica. Lo único que cambia es la lista de columnas. Una idea se queda solo si mejora el resultado de forma consistente, no empeora ningún modelo y se conoce en el momento de decidir.

El notebook solo reúne y muestra resultados. El código está en `src/proyecto/features/build.py` (las variables), `src/proyecto/models/train.py` (el `Pipeline`) y `src/proyecto/models/evaluate.py` (la medición y la regla), cada uno con sus pruebas en `tests/`. La especificación es `especificaciones/04-features.md` y las decisiones quedan en `reports/04_features.md`.

## Tres advertencias
1. **Datos.** Se usa solo el entrenamiento (24.000 clientes) y la prueba no se abre. Esa prueba ya se miró en el EDA y se evaluó en el notebook de selección de modelo, así que no sirve como evaluación final independiente (revisión H1).
2. **El 5 a 1.** Es una hipótesis del grupo, no un costo medido por el banco. Por eso se habla de *penalización relativa* y nunca de dinero o ahorro.
3. **Qué tan firme es el resultado.** El paquete de variables se elige con los mismos datos con que se mide, así que su mejora sale algo optimista. Para limitarlo se repite la comparación con otros pliegues (sección 6). Una validación cruzada anidada lo reduciría más, pero todavía no está hecha.

## Palabras clave
| Término | Significado simple |
|---|---|
| **Variable (feature)** | Una columna nueva calculada con las que ya existen |
| **Pliegue** | Cada vez que se entrena con una parte de los datos y se mide con otra. 5 pliegues × 3 repeticiones dan 15 mediciones por variante |
| **Referencia** | Las 23 variables actuales, sin ideas nuevas. Todo se compara contra ella |
| **Mejora pareada** | Mejora media del AUC dividida entre cuánto varía esa mejora de un pliegue a otro. Se compara cada pliegue consigo mismo, así que la dificultad propia del pliegue no cuenta. Es la regla vigente |
| **Mejora / ruido** | La regla original: mejora media dividida entre cuánto varía el AUC de la variante entre pliegues. Se muestra solo para comparar |
| **Penalización relativa** | Con la hipótesis 5 a 1, el castigo medio por cliente por equivocarse. Menor es mejor. No es dinero |


## 0. Cargar los datos de entrenamiento
**Qué hace la celda:** lee `data/processed/train.joblib` (lo genera el notebook 03), comprueba que ningún cliente se repite y separa los predictores del objetivo con `make_xy` (que quita `ID` y objetivo). **No se carga la prueba.**

In [2]:
import joblib
import numpy as np
import pandas as pd

from proyecto.config import ID_COL, PROCESSED_DIR, TARGET
from proyecto.data.split import assert_unique_ids, make_xy

train = joblib.load(PROCESSED_DIR / "train.joblib")
assert_unique_ids(train)                      # partir por fila sigue siendo válido
X, y = make_xy(train)
y = y.to_numpy()
print(f"entrenamiento: {X.shape[0]:,} filas x {X.shape[1]} predictores (la prueba NO se abre)")
print(f"tasa de incumplimiento: {y.mean():.2%}")


entrenamiento: 24,000 filas x 23 predictores (la prueba NO se abre)
tasa de incumplimiento: 22.12%


## 1. Mis ideas, antes de medirlas
Todas se calculan fila por fila con lo que el banco **ya sabe al decidir** (historial de 6 meses y límite de crédito). Ninguna usa el objetivo ni aprende de los datos, así que no hay fuga (lo comprueban `test_features_do_not_use_the_target` y el resto de `tests/test_features.py`).

| # | Variable | Cómo se calcula | Por qué podría ayudar | ¿Se conoce al predecir? |
|---|---|---|---|---|
| 1 | `utilization_last`, `utilization_mean`, `utilization_max` | `BILL_AMT1`, media y máximo de las 6 facturas, sobre `LIMIT_BAL` | Una tarjeta casi llena deja menos margen; el monto solo no dice nada sin el límite | ☒ sí |
| 2 | `pending_balance_last` | `(BILL_AMT2 − PAY_AMT1) / LIMIT_BAL` | Lo que queda sin pagar tras el último pago | ☒ sí |
| 3 | `delay_streak` | Meses seguidos con atraso desde `PAY_0` hacia atrás | Distingue un atraso reciente de uno viejo y superado | ☒ sí |
| 4 | `delay_severity`, `severe_delay_months`, `n_delayed_months` | Suma de meses de atraso; meses con ≥ 2; meses con algún atraso | Resumen del historial en un número (`n_delayed_months` tiene la correlación más alta con el objetivo: 0,40) | ☒ sí |
| 5 | `n_zero_payments` | Meses con `PAY_AMT = 0` | Un cliente que no paga nada es una señal directa | ☒ sí |
| 6 | `n_minimum_payments` | Meses (de 5) con pago ≤ 5 % de una factura positiva | Quien paga solo el mínimo acumula deuda. **El 5 % es un supuesto, no un dato** | ☒ sí |
| 7 | `log_limit` | `ln(LIMIT_BAL)` | Acorta la cola larga del límite; ayuda a los modelos lineales | ☒ sí |
| 8 | Sin `SEX` / sin `PAY_0` | Quitar la columna | Equidad y dependencia del último estado de pago | ☒ sí |

**Regla de decisión.** Una idea o un paquete se queda si cumple las tres condiciones:
1. En al menos un modelo, la **mejora pareada es mayor que 1** y la variante gana en **12 de los 15 pliegues o más**.
2. **No empeora en promedio** en ningún modelo.
3. Se conoce al momento de predecir.

> **Historia de la regla, para ser transparentes.** La primera versión usaba «mejora / ruido» (el ruido era la variación del AUC entre pliegues). Con ella, el paquete E no se confirmó en una de dos repeticiones con otras semillas, aunque su mejora fue casi idéntica en las tres (+0,0088, +0,0088 y +0,0089) y ganó en los 45 pliegues de la logística. La causa es que la variación entre pliegues cambia mucho de una semilla a otra (0,006 a 0,011) y afecta por igual a la referencia y a la variante. Por eso el grupo pasó a la mejora pareada el 2026-10-31, **después de ver ese resultado**. Las dos cifras se muestran abajo y la especificación guarda el detalle.
>
> Las ideas que una medición preliminar ya descartaba (`max_atraso`, `atraso_medio`, `tendencia_pay`, razones de pago, tendencia de factura…) no se implementaron; sus cifras están solo en la especificación.

## 2. Validación común
**Qué hace la celda:** arma los pliegues una sola vez (5 estratificados, repetidos 3 veces, semilla 42) y los reutiliza en todas las comparaciones. Comprueba que en cada pliegue ningún cliente queda en los dos lados y que la tasa de incumplimiento se mantiene.

In [3]:
from proyecto.models.evaluate import repeated_folds

folds = repeated_folds(X, y, n_splits=5, n_repeats=3, seed=42)
for train_idx, valid_idx in folds:
    assert set(train_idx).isdisjoint(valid_idx)                  # ningún cliente en los dos lados
    assert abs(y[valid_idx].mean() - y.mean()) < 0.005           # misma tasa de incumplimiento
print(f"{len(folds)} pliegues de ~{len(folds[0][1]):,} clientes de validación, todos estratificados")


15 pliegues de ~4,800 clientes de validación, todos estratificados


## 3. Pipeline base y métricas
**Qué hace la celda:** define los dos modelos de referencia con parámetros fijos (aquí no se optimiza nada; eso es de la etapa 6) y la forma de armar su `Pipeline` con `build_pipeline`:
- **LightGBM** (gradient boosting): el candidato principal según el EDA y el notebook 01. No necesita escalar los datos.
- **Regresión logística**: el contraste explicable. Estandariza los números y trata `PAY_*` como estados (one-hot), porque no son cantidades.

Todo lo que se aprende de los datos (medianas, escalas, categorías) se ajusta dentro de cada pliegue, solo con su parte de entrenamiento.

**Métricas.** El ROC AUC es la principal porque ordena a los clientes de menor a mayor riesgo. El AUC-PR y la penalización relativa son de control. La penalización usa un umbral fijo desde el principio, 1/6 ≈ 0,167, que es el punto de indiferencia de la matriz 5 a 1 si las probabilidades están calibradas. No se elige mirando cada pliegue de validación.

In [4]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression

from proyecto.models.train import build_pipeline

LGBM_PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=50,
                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1, n_jobs=4)

MODELOS = {
    "LightGBM": lambda features, drop: build_pipeline(LGBMClassifier(**LGBM_PARAMS), features, drop),
    "Logística": lambda features, drop: build_pipeline(LogisticRegression(C=0.5, max_iter=2000), features, drop,
                                                       linear=True, pay_as_category=True),
}
METRICA = "roc_auc"
print("modelos:", list(MODELOS), "| métrica principal:", METRICA)


modelos: ['LightGBM', 'Logística'] | métrica principal: roc_auc


## 4. El experimento
**Qué hace la celda:** para cada modelo mide la referencia y una variante por idea, más dos paquetes y las dos variantes en que se quita una columna. Cada medición entrena un `Pipeline` nuevo en cada uno de los 15 pliegues, así que tarda varios minutos.

Los paquetes son **E** (5 variables, el candidato de la medición preliminar) y **D** (10 variables, para ver si usar más ayuda).

In [5]:
from proyecto.features.build import PACKAGE_D, PACKAGE_E
from proyecto.models.evaluate import score_folds

VARIANTES = {   # nombre -> (variables que se añaden, columnas que se quitan)
    "referencia (23 variables)": ((), ()),
    "+ utilización (utilization_last, utilization_mean, utilization_max)": (("utilization_last", "utilization_mean", "utilization_max"), ()),
    "+ pending_balance_last": (("pending_balance_last",), ()),
    "+ delay_streak": (("delay_streak",), ()),
    "+ delay_severity": (("delay_severity",), ()),
    "+ severe_delay_months": (("severe_delay_months",), ()),
    "+ n_delayed_months": (("n_delayed_months",), ()),
    "+ n_zero_payments": (("n_zero_payments",), ()),
    "+ n_minimum_payments": (("n_minimum_payments",), ()),
    "+ log_limit": (("log_limit",), ()),
    "paquete E (5 variables)": (PACKAGE_E, ()),
    "paquete D (10 variables)": (PACKAGE_D, ()),
    "sin SEX": ((), ("SEX",)),
    "sin PAY_0": ((), ("PAY_0",)),
}

resultados = {modelo: {} for modelo in MODELOS}
for modelo, fabrica in MODELOS.items():
    for nombre, (features, drop) in VARIANTES.items():
        resultados[modelo][nombre] = score_folds(lambda f=features, d=drop, fab=fabrica: fab(f, d), X, y, folds)
    print(f"{modelo}: {len(VARIANTES)} variantes medidas")


LightGBM: 14 variantes medidas


Logística: 14 variantes medidas


## 5. Mejora de cada variante
**Qué hace la celda:** compara cada variante con la referencia, pliegue por pliegue, en cada modelo. Para cada una muestra el AUC medio, la mejora, la **mejora pareada** (la regla vigente), la **mejora / ruido** (la regla original, solo para comparar) y en cuántos de los 15 pliegues gana. Las dos últimas columnas dan la decisión con cada regla.

> Una mejora de 0,002 de AUC parece poca frente a una variación entre pliegues de ±0,007. Por eso se mira también en cuántos pliegues gana y se compara cada pliegue consigo mismo.

In [6]:
from proyecto.models.evaluate import decide, improvement_over_noise

def tabla_mejora(resultados, referencia="referencia (23 variables)", metrica=METRICA):
    filas, comparaciones = [], {}
    for nombre in VARIANTES:
        fila = {"variante": nombre}
        comparaciones[nombre] = {}
        for modelo in MODELOS:
            ref, var = resultados[modelo][referencia][metrica], resultados[modelo][nombre][metrica]
            r = improvement_over_noise(ref, var)
            comparaciones[nombre][modelo] = r
            fila[f"{modelo}: AUC"] = var.mean()
            fila[f"{modelo}: mejora"] = r["gain"]
            fila[f"{modelo}: pareada"] = r["paired_gain"]
            fila[f"{modelo}: mejora/ruido"] = r["gain_over_noise"]
            fila[f"{modelo}: pliegues"] = f"{r['folds_improved']}/{r['n_folds']}"
        if nombre != referencia:
            fila["decisión (pareada)"] = "conservar" if decide(comparaciones[nombre])[0] else "descartar"
            fila["decisión (original)"] = "conservar" if decide(comparaciones[nombre], key="gain_over_noise")[0] else "descartar"
        else:
            fila["decisión (pareada)"] = fila["decisión (original)"] = "—"
        filas.append(fila)
    return pd.DataFrame(filas).set_index("variante"), comparaciones

tabla, comparaciones = tabla_mejora(resultados)
pd.set_option("display.width", 300)
display(tabla.round(4))


,LightGBM: AUC,LightGBM: mejora,LightGBM: pareada,LightGBM: mejora/ruido,LightGBM: pliegues,Logística: AUC,Logística: mejora,Logística: pareada,Logística: mejora/ruido,Logística: pliegues,decisión (pareada),decisión (original)
variante,,,,,,,,,,,,
referencia (23 variables),0.7853,0.0000,0.0000,0.0000,0/15,0.7697,0.0000,0.0000,0.0000,0/15,—,—
"+ utilización (utilization_last, utilization_mean, utilization_max)",0.7872,0.0019,0.9688,0.3190,12/15,0.7704,0.0007,1.0932,0.1197,13/15,conservar,descartar
+ pending_balance_last,0.7864,0.0011,0.5721,0.1714,9/15,0.7703,0.0006,0.8779,0.0972,12/15,descartar,descartar
+ delay_streak,0.7851,-0.0002,-0.1709,-0.0214,6/15,0.7718,0.0021,2.1482,0.3356,15/15,descartar,descartar
+ delay_severity,0.7855,0.0002,0.1182,0.0308,8/15,0.7697,-0.0000,-0.2093,-0.0042,4/15,descartar,descartar
+ severe_delay_months,0.7858,0.0005,0.3313,0.0698,7/15,0.7697,-0.0001,-0.6151,-0.0109,6/15,descartar,descartar
+ n_delayed_months,0.7849,-0.0004,-0.2510,-0.0567,7/15,0.7697,-0.0001,-0.4131,-0.0094,5/15,descartar,descartar
+ n_zero_payments,0.7856,0.0003,0.2631,0.0404,8/15,0.7743,0.0045,2.8579,0.8172,15/15,conservar,descartar
+ n_minimum_payments,0.7852,-0.0001,-0.0651,-0.0085,8/15,0.7695,-0.0002,-0.4513,-0.0355,5/15,descartar,descartar


**Métricas de control.** Lo mismo con el AUC-PR y con la penalización relativa. En la penalización menor es mejor, así que una «Δ» positiva significa que bajó.

In [7]:
control = []
for modelo in MODELOS:
    for nombre in VARIANTES:
        d = resultados[modelo][nombre]
        ref = resultados[modelo]["referencia (23 variables)"]
        control.append({"modelo": modelo, "variante": nombre,
                        "AUC-PR": d["avg_precision"].mean(),
                        "Δ AUC-PR": (d["avg_precision"] - ref["avg_precision"]).mean(),
                        "penalización": d["penalty"].mean(),
                        "Δ penalización (positivo = mejor)": (ref["penalty"] - d["penalty"]).mean()})
display(pd.DataFrame(control).set_index(["modelo", "variante"]).round(4))


AUC-PR  Δ AUC-PR  penalización  Δ penalización (positivo = mejor)
modelo    variante                                                                                                             
LightGBM  referencia (23 variables)                           0.5606    0.0000        0.5471                             0.0000
          + utilización (utilization_last, utilization_me...  0.5635    0.0029        0.5447                             0.0024
          + pending_balance_last                              0.5626    0.0019        0.5451                             0.0020
          + delay_streak                                      0.5614    0.0008        0.5459                             0.0012
          + delay_severity                                    0.5612    0.0005        0.5437                             0.0034
          + severe_delay_months                               0.5606    0.0000        0.5424                             0.0047
          + n_delayed_months                                  0.5601   -0.0005        0.5468                             0.0003
          + n_zero_payments                                   0.5610    0.0003        0.5470                             0.0001
          + n_minimum_payments                                0.5611    0.0004        0.5461                             0.0009
          + log_limit                                         0.5610    0.0004        0.5444                             0.0027
          paquete E (5 variables)                             0.5635    0.0028        0.5456                             0.0014
          paquete D (10 variables)                            0.5633    0.0027        0.5409                             0.0062
          sin SEX                                             0.5609    0.0002        0.5465                             0.0006
          sin PAY_0                                           0.5015   -0.0592        0.5848                            -0.0378
Logística referencia (23 variables)                           0.5440    0.0000        0.5679                             0.0000
          + utilización (utilization_last, utilization_me...  0.5432   -0.0008        0.5668                             0.0011
          + pending_balance_last                              0.5431   -0.0008        0.5682                            -0.0003
          + delay_streak                                      0.5464    0.0024        0.5665                             0.0014
          + delay_severity                                    0.5441    0.0001        0.5683                            -0.0004
          + severe_delay_months                               0.5443    0.0003        0.5685                            -0.0006
          + n_delayed_months                                  0.5438   -0.0001        0.5682                            -0.0003
          + n_zero_payments                                   0.5487    0.0048        0.5630                             0.0049
          + n_minimum_payments                                0.5440    0.0001        0.5668                             0.0011
          + log_limit                                         0.5448    0.0008        0.5672                             0.0007
          paquete E (5 variables)                             0.5483    0.0043        0.5538                             0.0141
          paquete D (10 variables)                            0.5487    0.0048        0.5552                             0.0127
          sin SEX                                             0.5441    0.0002        0.5678                             0.0001
          sin PAY_0                                           0.4837   -0.0602        0.6181                            -0.0502

## 6. Repetición con otros pliegues
**Por qué:** el paquete se eligió mirando los mismos 15 valores con que se mide, así que su mejora sale algo optimista. Aquí se repite solo la comparación referencia contra paquete con **pliegues nuevos** (semillas 7 y 99). Si el paquete sigue ganando en pliegues que no vio al elegirlo, la conclusión es más firme. Estas semillas son las mismas que mostraron el problema de la regla original, así que no verifican de forma independiente la regla nueva. No reemplaza una validación anidada ni es una evaluación independiente, porque los 24.000 clientes son los mismos.

In [8]:
PAQUETE_CANDIDATO = PACKAGE_E
confirmacion = {}
for semilla in (7, 99):
    pliegues_nuevos = repeated_folds(X, y, n_splits=5, n_repeats=3, seed=semilla)
    por_modelo = {}
    for modelo, fabrica in MODELOS.items():
        ref = score_folds(lambda: fabrica((), ()), X, y, pliegues_nuevos)[METRICA]
        var = score_folds(lambda: fabrica(PAQUETE_CANDIDATO, ()), X, y, pliegues_nuevos)[METRICA]
        por_modelo[modelo] = improvement_over_noise(ref, var)
    confirmacion[semilla] = (por_modelo, decide(por_modelo)[0], decide(por_modelo, key="gain_over_noise")[0])
    print(f"semilla {semilla}: " + " | ".join(f"{m}: Δ={r['gain']:+.4f}, pareada={r['paired_gain']:.2f}, mejora/ruido={r['gain_over_noise']:.2f}, "
                                              f"pliegues={r['folds_improved']}/{r['n_folds']}" for m, r in por_modelo.items())
          + f"  ->  regla pareada: {'conserva' if confirmacion[semilla][1] else 'descarta'} | regla original: {'conserva' if confirmacion[semilla][2] else 'descarta'}")


semilla 7: LightGBM: Δ=+0.0022, pareada=1.53, mejora/ruido=0.28, pliegues=14/15 | Logística: Δ=+0.0088, pareada=2.53, mejora/ruido=1.22, pliegues=15/15  ->  regla pareada: conserva | regla original: conserva


semilla 99: LightGBM: Δ=+0.0029, pareada=1.47, mejora/ruido=0.32, pliegues=14/15 | Logística: Δ=+0.0089, pareada=3.30, mejora/ruido=0.81, pliegues=15/15  ->  regla pareada: conserva | regla original: descarta


## 7. Decisiones
Las cifras salen de las tablas anteriores. Una idea descartada con su cifra también es un resultado: deja escrito lo que ya se probó.

**Cómo se armó E.** El paquete E se definió en una medición preliminar, antes de aplicar la regla; la regla se aplicó después al paquete completo. Por eso hay ideas que «no pasan solas» y aun así están en E: la columna «Decisión» indica quién contribuye, no qué entra en E.

**Cómo leer la tabla.** Cada celda de modelo muestra la mejora del AUC, entre paréntesis la mejora pareada y en cuántos de los 15 pliegues gana. «Conservar» y «descartar» aplican la regla de la sección 1.

| Idea | LightGBM | Regresión logística | Decisión | Por qué |
|---|---|---|---|---|
| Utilización (`utilization_last`, `utilization_mean`, `utilization_max`) | +0,0019 (0,97; 12/15) | +0,0007 (1,09; 13/15) | Conservar | Pasa en la logística y no empeora LightGBM. La mejora es pequeña. En el paquete E entra solo `utilization_mean` |
| `pending_balance_last` | +0,0011 (0,57; 9/15) | +0,0006 (0,88; 12/15) | No pasa sola | Se queda dentro de E porque, junto con las demás, suma en LightGBM |
| `delay_streak` | −0,0002 (−0,17; 6/15) | +0,0021 (2,15; 15/15) | No pasa sola | LightGBM baja 0,0002, que es ruido. Ayuda a la logística y por eso va en E |
| `n_zero_payments` | +0,0003 (0,26; 8/15) | +0,0045 (2,86; 15/15) | Conservar | Es la mejor idea individual para la logística |
| `log_limit` | −0,0004 (−0,31; 6/15) | +0,0030 (2,47; 15/15) | No pasa sola | Solo ayuda al modelo lineal; a LightGBM le da igual. Va en E |
| `delay_severity`, `severe_delay_months`, `n_delayed_months` | ≈ 0 | ≈ 0 | Descartar | `PAY_*` ya contiene esa información y LightGBM la reconstruye solo |
| `n_minimum_payments` | −0,0001 (−0,07; 8/15) | −0,0002 (−0,45; 5/15) | Descartar | No aporta, y además depende de un 5 % supuesto |
| **Paquete E** (5 variables) | **+0,0017 (0,87; 13/15)** | **+0,0088 (5,00; 15/15)** | **Conservar** | Cumple en la logística, no empeora LightGBM y se confirmó con otros pliegues |
| Paquete D (10 variables) | +0,0032 (1,67; 15/15) | +0,0084 (3,97; 15/15) | Alternativa | Pasa en los dos modelos, pero no se repitió con otros pliegues (no se midió). Ver abajo |
| Sin `SEX` | −0,0010 (−1,22; 2/15) | −0,0009 (−1,01; 3/15) | No mejora | Quitarla cuesta unos 0,001 de AUC, de forma consistente |
| Sin `PAY_0` | −0,0295 (−6,03; 0/15) | −0,0340 (−5,79; 0/15) | No mejora | Es la columna de la que más depende el modelo |

**Repetición con otros pliegues (paquete E).** Con la semilla 7, la logística mejora +0,0088 (pareada 2,53; 15/15). Con la semilla 99, mejora +0,0089 (pareada 3,30; 15/15). Con la regla pareada se conserva en las dos. Con la regla original se conserva en la semilla 7 y no en la 99 (mejora / ruido 0,81), y esa diferencia es la que motivó el cambio de regla.

### Conclusiones
1. **El paquete final es E:** `utilization_mean`, `pending_balance_last`, `delay_streak`, `n_zero_payments` y `log_limit` (constante `FINAL_FEATURES`). Son cinco variables sencillas que se calculan con datos que el banco ya tiene al decidir.
2. **El beneficio es sobre todo para el modelo explicable.** La regresión logística sube su AUC de 0,7697 a 0,7785, su AUC-PR de 0,544 a 0,548 y baja su penalización relativa de 0,568 a 0,554. Así se acerca bastante a LightGBM.
3. **Para LightGBM la ganancia es pequeña:** +0,0017 de AUC, +0,003 de AUC-PR y 0,001 menos de penalización. No cambia la elección del modelo. El torneo (notebook 05) decidirá con ajuste de parámetros.
4. **El paquete D es una alternativa razonable si gana LightGBM.** Mejora un poco más a LightGBM (+0,0032), pero necesita diez variables y no se confirmó con otros pliegues. Se deja anotado para revisarlo en el torneo y no se elige aquí, para no cambiar de candidato después de mirar los resultados.
5. **`PAY_0` domina el modelo.** Sin ella el AUC cae unos 0,03 en ambos modelos. Debe quedar dicho en el informe técnico.
6. **`SEX` aporta muy poco, pero no nada.** Excluirla por equidad cuesta unos 0,001 de AUC. Es una decisión de principio del grupo, no de rendimiento.

### Límites que hay que tener presentes
- **La regla pareada no mira el tamaño de la mejora.** Una mejora de +0,0007 (utilización en la logística) pasa porque es consistente, aunque en la práctica casi no se nota. Por eso el peso de la conclusión está en el paquete E y no en cada idea.
- **La regla se cambió después de ver un resultado.** Se justifica, pero queda registrado: la regla original no confirmó el paquete en la semilla 99.
- **El paquete se eligió con estos mismos datos.** La confirmación con otros pliegues ayuda, pero no es una validación anidada ni una evaluación independiente. Las cifras son algo optimistas.
- **El 5 a 1 y el umbral 1/6** siguen siendo hipótesis; la penalización relativa no es dinero.

In [9]:
# El paquete que usa el proyecto debe ser el que se confirmó con otros pliegues; si no, el notebook se detiene
from proyecto.features.build import FINAL_FEATURES
assert tuple(FINAL_FEATURES) == tuple(PAQUETE_CANDIDATO), "FINAL_FEATURES no coincide con el paquete evaluado"
assert all(confirmacion[s][1] for s in confirmacion), "el paquete no se confirmó con la regla vigente en otros pliegues"
print("Paquete final:", list(FINAL_FEATURES))


Paquete final: ['utilization_mean', 'pending_balance_last', 'delay_streak', 'n_zero_payments', 'log_limit']


## Entregable de esta etapa

- [x] Cada idea tiene su fila con la mejora pareada, la mejora / ruido y una decisión (secciones 5 y 7).
- [x] Todas las variantes se midieron con los mismos pliegues (sección 2).
- [x] Ninguna variable usa información posterior a la decisión ni mira el objetivo (`tests/test_features.py`).
- [x] La prueba no se abrió.
- [x] El paquete elegido se repitió con otros pliegues (sección 6) y cumple la regla vigente. Con la regla original no se confirmó en la semilla 99, y eso queda escrito.
- [ ] Validación cruzada anidada: propuesta en la revisión H1, pendiente de aprobación del grupo.
- [ ] Revisión del código con `material-del-curso/kit/checklist-revision-ia.md`, para que cada integrante pueda explicar una función.
